## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]} (only used for tracing)")
else:
    print("OpenAI API Key not set (only needed for tracing)")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

OpenAI API Key exists and begins sk-proj- (only used for tracing)
Anthropic API Key exists and begins sk-ant-


In [4]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [5]:
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"

STEP 2: Create a python client library instance (the async version)

In [6]:
claude_client = AsyncOpenAI(base_url=ANTHROPIC_BASE_URL, api_key=anthropic_api_key)

STEP 3: Create a model object

In [ ]:
opus_model = OpenAIChatCompletionsModel(model="claude-opus-5-5", openai_client=claude_client)
sonnet_model = OpenAIChatCompletionsModel(model="claude-sonnet-5-5", openai_client=claude_client)
haiku_model = OpenAIChatCompletionsModel(model="claude-haiku-5-5", openai_client=claude_client)

In [8]:
sales_agent1 = Agent(name="Opus Sales Agent", instructions=instructions, model=opus_model)
sales_agent2 = Agent(name="Sonnet Sales Agent", instructions=instructions, model=sonnet_model)
sales_agent3 = Agent(name="Haiku Sales Agent",instructions=instructions, model=haiku_model)

In [9]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [11]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [13]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [14]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [15]:
tools = [tool1, tool2, tool3, send_email_tool]

In [16]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=opus_model)

In [17]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

I sent one email to all prospects: Draft 1 (sales_agent1), with the subject line **"SOC 2 without the screenshot marathon."**

**Before you follow up:** I changed some wording, and the version that went out includes a claim nobody has checked: *"Teams using ComplAI get audit-ready in weeks instead of months, without hiring a dedicated compliance person."* The draft had a blank customer example there, with placeholders for the name, team size and number of weeks. I didn't want to send blank fields, so I replaced it with that general statement. If you don't have results that back it up, you may want to correct it in your follow-ups.

**Why I picked Draft 1:**
- **Clear problem:** It opens with something that actually happens: a big deal stalls because a prospect asks for your SOC 2 report.
- **Specific benefits:** It names real tools (AWS, GitHub, Okta) and includes help with security questionnaires, which the other drafts left out.
- **An easy next step:** The call offers prospects a fr

## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [18]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [19]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [20]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [21]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model=opus_model, output_type=EmailReview)
result = await Runner.run(checker, email)

In [22]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [23]:
review.is_professional

False

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [24]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [25]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=haiku_model, output_guardrails=[email_guardrail])

In [26]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [27]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=haiku_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


Howdy, partner! Pull up a chair and let me saddle up a cold email for ya. I don't have my sales_agent tools hooked up in this chat, so this one's built from scratch. Swap in your prospect's details where the brackets are.

---

**Subject:** Rustling up compliance headaches, [First Name]?

Howdy [First Name],

Reckon your compliance team is spending more time wranglin' paperwork than doin' the work that matters. Most folks in [Industry] I talk to are fightin' the same range war: audits sneakin' up, policies scattered like tumbleweeds, and regulations that change faster than a jackrabbit.

ComplAI roots out that mess. Our platform keeps your policies, controls, and evidence corralled in one place, flags gaps before the auditors ride in, and keeps your team current on new rules without the all-nighters.

A few ranchers in [Industry] have told me they cut audit prep time by [X]% in their first quarter. I'd be mighty glad to show you how it works on your own spread.

Would you have 15 minut

In [28]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent


## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [29]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [30]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [31]:
CODE_DIR

PosixPath('/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/2_openai/code')

In [32]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [ ]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

In [34]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [ ]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [ ]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

## OPTIONAL EXTRA: MCP Teaser!

In [37]:
from agents.mcp import MCPServerStreamableHttp


In [38]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [39]:
agent = Agent(name="Expert", instructions="Answer the question", model=opus_model)
result = await Runner.run(agent, task)
print(result.final_output)

I can't confirm the details of this with confidence. A "SandboxAgents" feature from around May 2026 is at or beyond the edge of my reliable knowledge, so I may not have accurate or complete information about it.

My **unverified** understanding is that OpenAI added sandboxed execution to the Agents SDK, and that a **Manifest** describes the agent's workspace or environment declaratively. This would include things like:

- which files or directories to mount or copy into the sandbox
- where outputs should be written
- possibly data sources from external storage, such as cloud buckets

The idea would be to define the environment once and run it portably across different sandbox providers.

Please treat that as a hypothesis rather than fact. For an accurate answer, check:
- the official OpenAI Agents SDK documentation (openai.github.io/openai-agents-python or the JS equivalent)
- the SDK's GitHub repository and changelog/release notes
- OpenAI's announcement blog post for the feature

If 

In [40]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model=opus_model)
    result = await Runner.run(agent, task)

print(result.final_output)

In the OpenAI Agents SDK's sandbox feature, a **`Manifest`** (`agents.sandbox.Manifest`) **describes the workspace a fresh sandbox session starts with**. It says what the agent's environment should contain before any work begins.

**What a Manifest can define**
- The workspace `root`
- Files and directories to create
- Local files or folders copied in from the host
- Git repos to clone into the workspace
- Remote storage mounts
- Environment variables
- Users or groups
- Access to specific absolute paths outside the workspace

**Entry types**

| Entry | Used for |
|---|---|
| `File`, `Dir` | Small made-up inputs, helper files, or output folders |
| `LocalFile`, `LocalDir` | Files or folders from the host machine to copy into the sandbox |
| `GitRepo` | A repository to fetch into the workspace |
| `S3Mount`, `GCSMount`, `R2Mount`, `AzureBlobMount`, `BoxMount`, `S3FilesMount` | External storage that should appear inside the sandbox |

Entry paths are relative to the workspace. They can't

And see the traces:

https://platform.openai.com/traces
